In [ ]:
# %% Config
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image
from scipy.ndimage import zoom

# REPO = Path("../..").resolve()
REPO = Path("./").resolve()
CLS_DIR = REPO / "external" / "PanDerm" / "classification"
sys.path.insert(0, str(CLS_DIR))

# ---- paths to verify ----
CKPT_GAP  = REPO / "external" / "checkpoints5" / "checkpoint-best-gap-ha5.pth"
CKPT_CLS  = REPO / "external" / "checkpoints5" / "checkpoint-best-cls-ha5.pth"
IMG_DIR   = REPO / "data" / "HAM10000" / "images"
MASK_DIR  = REPO / "data" / "HAM10000_segmentations_lesion_tschandl"
SPLIT_CSV = REPO / "data" / "HAM10000" / "mel_nv" / "ham_mel_nv_clean_balanced_by_split.csv"

COL_ID    = "image_id"
COL_LABEL = "gt_label"
COL_DX    = "dx"
COL_SPLIT = "split"

IMG_EXT   = ".jpg"
MASK_SUFFIX = "_segmentation.png"

# ---- figure settings ----
OUT = REPO / "figures" / "slides"
OUT.mkdir(parents=True, exist_ok=True)

HERO_ID    = "ISIC_0024308"
INPUT_SIZE = 224
GAMMA      = 0.6
TOP_PCT    = 10
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"

C_GAP = "#4682B4"
C_CLS = "#FF8C00"

mpl.rcParams.update({
    "figure.dpi": 200,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.transparent": True,
    "font.size": 9,
    "axes.titlesize": 10,
})

print("device:", DEVICE)
for p in [CKPT_GAP, CKPT_CLS, IMG_DIR, MASK_DIR, SPLIT_CSV]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

device: cuda
OK   /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-gap-ha5.pth
OK   /storage/homefs/cn21m021/projects/master-thesis/external/checkpoints5/checkpoint-best-cls-ha5.pth
OK   /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/images
OK   /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000_segmentations_lesion_tschandl
OK   /storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/mel_nv/ham_mel_nv_clean_balanced_by_split.csv


In [2]:
# %% Discovery: what is inside the checkpoint
_ck = torch.load(CKPT_GAP, map_location="cpu", weights_only=False)

print("top level keys:", list(_ck.keys())[:12])

_state = _ck.get("model", _ck.get("module", _ck))
print("n tensors:", len(_state))

_head = [k for k in _state if "head" in k and k.endswith("weight")]
print("head keys:", _head)
for k in _head:
    print(f"  {k}: {tuple(_state[k].shape)}   -> num_classes = {_state[k].shape[0]}")

_pe = [k for k in _state if "patch_embed" in k]
print("patch_embed:", _pe[:4])
if "patch_embed.proj.weight" in _state:
    print("  patch size:", _state["patch_embed.proj.weight"].shape[-1])
if "pos_embed" in _state:
    print("  pos_embed:", tuple(_state["pos_embed"].shape))

_nblocks = len({k.split(".")[1] for k in _state if k.startswith("blocks.")})
print("n blocks:", _nblocks)
print("embed dim:", _state["blocks.0.attn.qkv.weight"].shape[1])
print("has fc_norm (mean pooling marker):",
      any(k.startswith("fc_norm") for k in _state))

if "args" in _ck:
    _a = _ck["args"]
    _a = vars(_a) if not isinstance(_a, dict) else _a
    keep = ["model", "input_size", "nb_classes", "use_mean_pooling",
            "imagenet_default_mean_and_std", "ha_lambda", "ha_loss_type",
            "ha_fp_weight", "data_path", "crop_pct"]
    print("\nsaved args:")
    print(json.dumps({k: str(_a.get(k)) for k in keep if k in _a}, indent=2))
else:
    print("\nno args saved in checkpoint")

top level keys: ['model', 'optimizer', 'epoch', 'scaler', 'args']
n tensors: 188
head keys: ['head.weight']
  head.weight: (2, 768)   -> num_classes = 2
patch_embed: ['patch_embed.proj.weight', 'patch_embed.proj.bias']
  patch size: 16
  pos_embed: (1, 197, 768)
n blocks: 12
embed dim: 768
has fc_norm (mean pooling marker): True

saved args:
{
  "model": "PanDerm_Base_FT",
  "input_size": "224",
  "nb_classes": "2",
  "ha_lambda": "5.0",
  "ha_loss_type": "paper_dice",
  "ha_fp_weight": "1.0"
}


In [6]:
# %% Check A: inspect every candidate label column
df = pd.read_csv(SPLIT_CSV)

cand = ["gt_label", "label", "label_2class", "binary_label"]
for c in cand:
    if c in df.columns:
        print(f"--- {c}  dtype={df[c].dtype}")
        print(pd.crosstab(df["dx"], df[c]))
        print()

--- gt_label  dtype=str
gt_label   MEL    NV
dx                  
mel       1113     0
nv           0  1113

--- label  dtype=int64
label     4     5
dx               
mel    1113     0
nv        0  1113

--- label_2class  dtype=int64
label_2class     0     1
dx                      
mel           1113     0
nv               0  1113

--- binary_label  dtype=int64
binary_label     0     1
dx                      
mel           1113     0
nv               0  1113



In [7]:
# %% Check B: pick the integer column and derive the mapping
COL_LABEL = "binary_label"        # <- set from Check A output

assert pd.api.types.is_integer_dtype(df[COL_LABEL]), \
    f"{COL_LABEL} is {df[COL_LABEL].dtype}, expected int"

_m = (df[["dx", COL_LABEL]]
      .drop_duplicates()
      .set_index("dx")[COL_LABEL]
      .to_dict())

MEL = int(_m["mel"])
NV  = int(_m["nv"])
assert {MEL, NV} == {0, 1}, f"unexpected labels {_m}"
print(f"MEL = {MEL}   NV = {NV}")

MEL = 0   NV = 1


In [8]:
# %% Path check
print(df[["image_rel_path", "mask_rel_path", "mask_found"]].head(3).to_string())
print("\nmask_found:", df["mask_found"].value_counts().to_dict())

DATA_ROOT = REPO / "data"
r = df.iloc[0]
print("\nresolves:", (DATA_ROOT / r["image_rel_path"]).exists(),
                     (DATA_ROOT / r["mask_rel_path"]).exists())

            image_rel_path                                                            mask_rel_path  mask_found
0  images/ISIC_0024459.jpg  ../HAM10000_segmentations_lesion_tschandl/ISIC_0024459_segmentation.png           1
1  images/ISIC_0024571.jpg  ../HAM10000_segmentations_lesion_tschandl/ISIC_0024571_segmentation.png           1
2  images/ISIC_0024624.jpg  ../HAM10000_segmentations_lesion_tschandl/ISIC_0024624_segmentation.png           1

mask_found: {1: 2226}

resolves: False False


In [9]:
# %% Path check
DATA_ROOT = REPO / "data" / "HAM10000"

r = df.iloc[0]
ip = (DATA_ROOT / r["image_rel_path"]).resolve()
mp = (DATA_ROOT / r["mask_rel_path"]).resolve()
print(ip, ip.exists())
print(mp, mp.exists())

/storage/homefs/cn21m021/projects/master-thesis/data/HAM10000/images/ISIC_0024459.jpg True
/storage/homefs/cn21m021/projects/master-thesis/data/HAM10000_segmentations_lesion_tschandl/ISIC_0024459_segmentation.png True


In [11]:
# %% Which nevi survived the undersample
full = pd.read_csv(REPO / "data" / "HAM10000" / "HAM10000.csv")  # adjust
nv_all  = full[full.dx == "nv"]
nv_kept = set(df[df.dx == "nv"].image_id)

print("all NV:")
print(nv_all.dx_type.value_counts(normalize=True).round(3))
print("\nkept NV:")
print(nv_all[nv_all.image_id.isin(nv_kept)].dx_type.value_counts(normalize=True).round(3))

all NV:
dx_type
follow_up    0.552
histo        0.373
consensus    0.075
Name: proportion, dtype: float64

kept NV:
dx_type
follow_up    0.499
histo        0.432
consensus    0.069
Name: proportion, dtype: float64


In [12]:
# %% Check C: does the trained head agree with the CSV
# run after Cell 4. confirms the model was trained on this same mapping.
def verify_head_order(model, n=96):
    sub = df[df[COL_SPLIT] == "test"].sample(n, random_state=0)
    hit = {MEL: [0, 0], NV: [0, 0]}
    for _, r in sub.iterrows():
        img, _, gt = load_sample(r[COL_ID])
        with torch.no_grad():
            lg = model(to_tensor(img))
            lg = lg[0] if isinstance(lg, (tuple, list)) else lg
            lg = lg["logits"] if isinstance(lg, dict) else lg
        pred = int(lg.argmax(1).item())
        hit[int(gt)][0] += int(pred == int(gt))
        hit[int(gt)][1] += 1
    for k, name in [(MEL, "MEL"), (NV, "NV")]:
        c, t = hit[k]
        print(f"{name} (idx {k}): {c}/{t} correct = {c/max(t,1):.2f}")
    print("\nboth well above 0.5 means the head order matches the CSV.")
    print("both near 0.0 means the mapping is inverted.")

In [13]:
# %% Adapter
from timm.models import create_model
import modeling_finetune  # registers PanDerm architectures
from run_class_finetuning_ha import (
    build_attention_gradcam_map,
    unpack_model_outputs,
    _unwrap_model,
)

MODEL_NAME  = "panderm_base_patch16_224"   # set from Cell 2
NUM_CLASSES = 2

MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

_CACHE = {}

def load_model(ckpt_path, pooling):
    key = (str(ckpt_path), pooling)
    if key in _CACHE:
        return _CACHE[key]

    model = create_model(
        MODEL_NAME,
        pretrained=False,
        num_classes=NUM_CLASSES,
        drop_rate=0.0,
        drop_path_rate=0.0,
        attn_drop_rate=0.0,
        use_mean_pooling=(pooling == "gap"),
    )

    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    state = ck.get("model", ck.get("module", ck))
    state = {k.replace("module.", ""): v for k, v in state.items()}
    missing, unexpected = model.load_state_dict(state, strict=False)
    if missing:
        print(f"  missing: {missing[:6]}")
    if unexpected:
        print(f"  unexpected: {unexpected[:6]}")

    model.to(DEVICE)
    model.eval()
    for p in model.parameters():
        p.requires_grad_(True)   # attention must stay on the graph

    got = bool(getattr(model, "use_mean_pooling", None))
    print(f"loaded {ckpt_path.name}  pooling={pooling}  use_mean_pooling={got}")
    assert got == (pooling == "gap"), "pooling flag mismatch"

    _CACHE[key] = model
    return model


_LBL = df.set_index(COL_ID)[COL_LABEL].to_dict()

def load_sample(image_id):
    r = _ROW.loc[image_id]

    img = Image.open((DATA_ROOT / r["image_rel_path"]).resolve()).convert("RGB")
    img = img.resize((INPUT_SIZE, INPUT_SIZE), Image.BICUBIC)
    img_u8 = np.asarray(img, dtype=np.uint8)

    msk = Image.open((DATA_ROOT / r["mask_rel_path"]).resolve()).convert("L")
    msk = msk.resize((INPUT_SIZE, INPUT_SIZE), Image.NEAREST)
    mask = np.asarray(msk) > 127

    return img_u8, mask, int(r[COL_LABEL])


_MEAN_T = torch.tensor(MEAN).view(1, 3, 1, 1)
_STD_T  = torch.tensor(STD).view(1, 3, 1, 1)

def to_tensor(img_u8):
    x = torch.from_numpy(img_u8).permute(2, 0, 1).float().div_(255.0)
    x = x.unsqueeze(0)
    x = (x - _MEAN_T) / _STD_T
    return x.to(DEVICE)

ModuleNotFoundError: No module named 'modeling_finetune'

In [ ]:
# %% Smoke test
m_gap = load_model(CKPT_GAP, "gap")
img, mask, gt = load_sample(HERO_ID)

print("image:", img.shape, img.dtype)
print("mask :", mask.shape, mask.dtype, "coverage", f"{mask.mean():.3f}")
print("label:", gt, "MEL" if gt == MEL else "NV")

x = to_tensor(img)
print("tensor:", tuple(x.shape), f"[{x.min():.2f}, {x.max():.2f}]")

with torch.no_grad():
    o = m_gap(x, return_patch_tokens=True, store_attn=True, attn_layer=-1)
    lg, pt = unpack_model_outputs(o)
print("logits:", lg.detach().cpu().numpy().round(3))
print("probs :", torch.softmax(lg, 1).detach().cpu().numpy().round(3))
print("patch tokens:", tuple(pt.shape))
print("patch grid:", _unwrap_model(m_gap).patch_embed.patch_shape)

_unwrap_model(m_gap).clear_xai_state()
verify_head_order(m_gap)

In [ ]:
# %% CAM driver
def open_graph(model, x):
    _unwrap_model(model).clear_xai_state()
    out = model(x, return_patch_tokens=True, store_attn=True, attn_layer=-1)
    logits, _ = unpack_model_outputs(out)
    return logits

def cam(model, logits, cls_idx, ref_idx=None, gamma=0.0, stages=False):
    """
    ref_idx None  -> GradCAM on cls_idx
    ref_idx set   -> Finer-CAM on (y_c - gamma * y_d)
    """
    b = logits.shape[0]
    t = torch.full((b,), cls_idx, device=logits.device, dtype=torch.long)

    scalar = None
    if ref_idx is not None:
        scalar = (logits[:, cls_idx] - gamma * logits[:, ref_idx]).sum()

    r = build_attention_gradcam_map(
        model, logits, t,
        create_graph=False,
        target_logits=scalar,
        return_stages=stages,
    )
    if stages:
        mp, st = r
        return (mp.detach()[0].float().cpu().numpy(),
                {k: v[0].float().cpu().numpy() for k, v in st.items()})
    return r.detach()[0].float().cpu().numpy()

In [ ]:
# %% Helpers
def norm01(a):
    a = np.asarray(a, np.float32)
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def upsample(c, hw):
    c = np.asarray(c, np.float32)
    if c.shape == tuple(hw):
        return c
    return zoom(c, (hw[0] / c.shape[0], hw[1] / c.shape[1]), order=1)

def topk(c, pct=TOP_PCT):
    return c >= np.percentile(c, 100 - pct)

def panel(ax, img, c=None, mask=None, pct=None, title=None):
    ax.imshow(img)
    if c is not None:
        cn = norm01(upsample(c, img.shape[:2]))
        if pct is not None:
            cn = np.where(topk(cn, pct), cn, np.nan)
        ax.imshow(cn, cmap="jet", alpha=0.55, vmin=0, vmax=1)
    if mask is not None:
        ax.contour(mask.astype(float), [0.5], colors="white", linewidths=1.4)
    ax.set_xticks([])
    ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    if title:
        ax.set_title(title, pad=6)

def iou(a, b, hw, pct=TOP_PCT):
    ma = topk(norm01(upsample(a, hw)), pct)
    mb = topk(norm01(upsample(b, hw)), pct)
    return (ma & mb).sum() / ((ma | mb).sum() + 1e-8)

def inside(c, mask, pct=TOP_PCT):
    t = topk(norm01(upsample(c, mask.shape)), pct)
    return (t & mask).sum() / (t.sum() + 1e-8)

In [ ]:
# %% FIG 1
img, mask, gt = load_sample(HERO_ID)
x = to_tensor(img)
m = load_model(CKPT_GAP, "gap")

with torch.enable_grad():
    lg = open_graph(m, x)
    cam_mel = cam(m, lg, MEL)
    cam_nv  = cam(m, lg, NV)      # same forward, retain_graph makes this valid
_unwrap_model(m).clear_xai_state()

hw   = img.shape[:2]
ov   = iou(cam_mel, cam_nv, hw)
ins  = inside(cam_mel, mask)
area = float(mask.mean())

fig, ax = plt.subplots(1, 3, figsize=(9.0, 3.2))
panel(ax[0], img, mask=mask,     title="Dermoscopic lesion")
panel(ax[1], img, cam_mel, mask, title='"Where is the melanoma?"')
panel(ax[2], img, cam_nv,  mask, title='"Where is the nevus?"')

fig.text(0.5, -0.02,
    f"Two opposite questions. Top {TOP_PCT} percent regions overlap at IoU {ov:.2f}. "
    f"{ins*100:.0f} percent of melanoma evidence falls inside a lesion "
    f"covering {area*100:.0f} percent of the frame.",
    ha="center", fontsize=8.5, style="italic")

fig.savefig(OUT / "fig1_interpretation_gap.png")
print(f"IoU {ov:.3f} | inside {ins:.3f} | mask area {area:.3f}")

In [ ]:
# %% FIG 2
with torch.enable_grad():
    lg = open_graph(m, x)
    _, st = cam(m, lg, int(gt), stages=True)
_unwrap_model(m).clear_xai_state()

specs = [
    ("attention", "1. Stored attention",        "viridis",  False),
    ("gradient",  "2. Gradient of the logit",   "coolwarm", False),
    ("product",   "3. Attention x gradient",    "magma",    False),
    ("cam",       "4. ReLU, reshape, upsample", None,       True),
]

for i, (k, title, cmap, as_overlay) in enumerate(specs, 1):
    f, a = plt.subplots(figsize=(2.7, 2.7))
    if as_overlay:
        panel(a, img, st[k], mask, title=title)
    else:
        a.imshow(norm01(st[k]), cmap=cmap)
        a.set_xticks([])
        a.set_yticks([])
        a.set_title(title, pad=6)
        for s in a.spines.values():
            s.set_visible(False)
    f.savefig(OUT / f"fig2_stage{i}.png")
    plt.close(f)

print("saved 4 stage PNGs")

In [ ]:
# %% FIG 4
tgt = int(gt)
ref = NV if tgt == MEL else MEL

with torch.enable_grad():
    lg  = open_graph(m, x)
    c_t = cam(m, lg, tgt)
    c_r = cam(m, lg, ref)
    c_f = cam(m, lg, tgt, ref_idx=ref, gamma=GAMMA)
_unwrap_model(m).clear_xai_state()

c_d = np.clip(norm01(c_t) - norm01(c_r), 0, None)

cols = [
    (None, "Lesion and mask"),
    (c_t,  "GradCAM target"),
    (c_r,  "GradCAM reference"),
    (c_d,  "Naive difference"),
    (c_f,  f"Finer-CAM  $\\gamma$ = {GAMMA}"),
]

fig, ax = plt.subplots(1, 5, figsize=(13.5, 3.0))
for a, (c, t) in zip(ax, cols):
    panel(a, img, c, mask, None if c is None else TOP_PCT, title=t)

fig.text(0.5, -0.03,
    f"All maps thresholded to the top {TOP_PCT} percent of activation.",
    ha="center", fontsize=8, style="italic")
fig.savefig(OUT / "fig4_explanation_panel.png")

print("naive diff vs finer IoU:", round(iou(c_d, c_f, hw), 3))

In [ ]:
# %% FIG 3
rev = pd.read_csv(REPO / "results" / "derm_pooling_review.csv")
order = ["gap", "tie", "cls"]
cnt = rev["verdict"].value_counts().reindex(order).fillna(0).astype(int)
n = int(cnt.sum())

fig, ax = plt.subplots(figsize=(6.2, 1.7))
left = 0
for k, col, lab in zip(order,
                       [C_GAP, "#BDBDBD", C_CLS],
                       ["GAP clearer", "Both acceptable", "CLS clearer"]):
    v = int(cnt[k])
    if v:
        ax.barh(0, v, left=left, color=col, edgecolor="white", height=0.55)
        ax.text(left + v / 2, 0, f"{lab}\n{v}",
                ha="center", va="center",
                color="black" if k == "tie" else "white",
                fontsize=8.5, fontweight="bold")
    left += v

ax.set_xlim(0, n)
ax.set_ylim(-0.5, 0.5)
ax.set_xticks([])
ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)
ax.set_title(f"Dermatologist preference across {n} reviewed cases", pad=8)
fig.savefig(OUT / "fig3_pooling_verdict.png")

In [ ]:
# %% FIG 3b  one worked example
EX = rev.loc[rev.verdict == "gap", "case_id"].iloc[0]
img_e, mask_e, gt_e = load_sample(EX)
xe = to_tensor(img_e)

m_cls = load_model(CKPT_CLS, "cls")

with torch.enable_grad():
    c_g = cam(m_gap, open_graph(m_gap, xe), int(gt_e))
_unwrap_model(m_gap).clear_xai_state()

with torch.enable_grad():
    c_c = cam(m_cls, open_graph(m_cls, xe), int(gt_e))
_unwrap_model(m_cls).clear_xai_state()

fig, ax = plt.subplots(1, 3, figsize=(7.6, 2.9))
panel(ax[0], img_e, mask=mask_e,             title="Lesion and mask")
panel(ax[1], img_e, c_c, mask_e, TOP_PCT,    title="CLS pooling")
panel(ax[2], img_e, c_g, mask_e, TOP_PCT,    title="GAP pooling")
ax[1].title.set_color(C_CLS)
ax[2].title.set_color(C_GAP)
fig.savefig(OUT / "fig3b_pooling_example.png")